In [2]:
from common import get_spark, LAKE
spark = get_spark("SahlPay-P2-RDD")
sc = spark.sparkContext
print("Spark", spark.version, "| master:", sc.master, "| cœurs:", sc.defaultParallelism)

Spark 3.5.1 | master: spark://spark-master:7077 | cœurs: 2


26/11/24 16:49:02 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [4]:
import json, time

PATH = f"{LAKE}/bronze/history/transactions_history.jsonl"

t0 = time.time()
raw    = sc.textFile(PATH)                          # transformation (lazy)
events = raw.map(json.loads)                        # transformation
# Le producteur injecte ~2 % de doublons (même tx_id + status) : on les retire
completed = (events.filter(lambda e: e["status"] == "COMPLETED")
                   .map(lambda e: (e["tx_id"], e))
                   .reduceByKey(lambda a, b: a)     # dédoublonnage : 1er shuffle
                   .values()
                   .cache())

# Q3 : montant total par ville
total_city = (completed.map(lambda e: (e["city"], e["amount_mad"]))
                       .reduceByKey(lambda a, b: a + b))

# Q4 : moyenne en UN seul passage : (somme, nombre)
sum_count = completed.map(lambda e: (e["city"], e["amount_mad"])).aggregateByKey(
    (0.0, 0),
    lambda acc, v: (acc[0] + v, acc[1] + 1),        # dans une partition
    lambda a, b: (a[0] + b[0], a[1] + b[1]))        # entre partitions
avg_city = sum_count.mapValues(lambda sc_: (round(sc_[0], 2), sc_[1], round(sc_[0] / sc_[1], 2)))

res = sorted(avg_city.collect(), key=lambda x: -x[1][0])   # ACTION
print(f"{'Ville':<12}{'Total MAD':>15}{'Nb':>8}{'Moyenne':>10}")
for city, (tot, n, avg) in res:
    print(f"{city:<12}{tot:>15,.2f}{n:>8}{avg:>10.2f}")

# Q5 : top 5 commerçants (paiements P2M validés)
top5 = (completed.filter(lambda e: e["type"] == "P2M")
                 .map(lambda e: (e["payee_id"], 1))
                 .reduceByKey(lambda a, b: a + b)
                 .takeOrdered(5, key=lambda kv: -kv[1]))   # ACTION
print(top5)
print("Temps RDD :", round(time.time() - t0, 2), "s")

Ville             Total MAD      Nb   Moyenne
Casablanca     1,771,958.41    4044    438.17
Rabat            905,406.01    2123    426.47
Marrakech        812,544.50    1421    571.81
Tanger           661,911.48    1866    354.72
Fes              496,963.56    1365    364.08
Oujda            326,583.26     419    779.43
Tetouan          315,692.32     399    791.21
Meknes           307,520.92     665    462.44
Agadir           293,650.67     919    319.53
Kenitra          274,320.34     807    339.93
[('M-001', 637), ('M-002', 372), ('M-003', 302), ('M-004', 230), ('M-005', 175)]
Temps RDD : 5.49 s


In [5]:
print(avg_city.toDebugString().decode())

(2) PythonRDD[36] at collect at /tmp/ipykernel_274/629414868.py:26 []
 |  MapPartitionsRDD[35] at mapPartitions at PythonRDD.scala:160 []
 |  ShuffledRDD[34] at partitionBy at <unknown>:0 []
 +-(2) PairwiseRDD[33] at aggregateByKey at /tmp/ipykernel_274/629414868.py:20 []
    |  PythonRDD[32] at aggregateByKey at /tmp/ipykernel_274/629414868.py:20 []
    |  PythonRDD[27] at RDD at PythonRDD.scala:53 []
    |      CachedPartitions: 2; MemorySize: 685.5 KiB; DiskSize: 0.0 B
    |  MapPartitionsRDD[26] at mapPartitions at PythonRDD.scala:160 []
    |  ShuffledRDD[25] at partitionBy at <unknown>:0 []
    +-(2) PairwiseRDD[24] at reduceByKey at /tmp/ipykernel_274/629414868.py:9 []
       |  PythonRDD[23] at reduceByKey at /tmp/ipykernel_274/629414868.py:9 []
       |  s3a://lake/bronze/history/transactions_history.jsonl MapPartitionsRDD[22] at textFile at <unknown>:0 []
       |  s3a://lake/bronze/history/transactions_history.jsonl HadoopRDD[21] at textFile at <unknown>:0 []


In [6]:
spark.stop()